<a href="https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/03_evaluate_and_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Evaluate and fix

You found the problems by reading traces. Now make them measurable: turn the traces you flagged
into a dataset, write a check for each problem, score the agent you built (v0), then score a
fixed version (v1) and compare.

**New in this lesson:** datasets from traces, code evaluators, an LLM judge that reads a photo,
trajectory evaluation with `agentevals`, experiments and the comparison view.

> **Need a key?** Store your LangSmith API key in Colab Secrets (🔑 in the left sidebar) as
> `LANGSMITH_API_KEY`, with **Notebook access** on. If you haven't yet, run
> **[00 · Setup](https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/00_setup.ipynb)** first.

In [ ]:
# --- snippet:setup v1 ---
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata:
    %pip install -qq --progress-bar off \
      "langchain~=1.4.3" \
      "langchain-openai~=1.6.7" \
      "langsmith~=0.14.4" \
      "agentevals~=0.0.9"

    try:
        os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise RuntimeError(
            "Add LANGSMITH_API_KEY in the 🔑 Secrets panel, turn on Notebook access, "
            "then run this cell again."
        ) from None
else:
    from dotenv import load_dotenv

    load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "hse-03-evaluate"

MODEL = "langsmith:openai/gpt-6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Turn your flagged traces into a dataset

In LangSmith, open the project **`hse-triage-prod`**:

1. Filter the traces to the ones you flagged: add a filter on the feedback key **`flag`**.
2. Select them all, then choose **Add to Dataset** and create a new dataset named
   **`hse-flagged`**.

Each example keeps the trace's input: the report, and its photo if it had one. Didn't flag
anything? The cell below falls back to a review set your instructor prepared.

In [ ]:
from langsmith import Client

client = Client()

DATASET = "hse-flagged" if client.has_dataset(dataset_name="hse-flagged") else "hse-triage-review"
examples = list(client.list_examples(dataset_name=DATASET))
print(f"Using {DATASET}: {len(examples)} examples")

---

## 2. The agent you built

The same agent as in 01, loaded in one cell. Run it.

In [ ]:
#@title The agent from 01 (v0) { display-mode: "form" }
import json
from difflib import get_close_matches
from typing import Literal

from langchain.agents import create_agent
from pydantic import BaseModel, Field

SITES = [
    {"site_id": "CC-03", "name": "Cedar Creek Pad 3", "kind": "well pad", "area": "North"},
    {"site_id": "CC-05", "name": "Cedar Creek Pad 5", "kind": "well pad", "area": "North"},
    {"site_id": "RM-TF", "name": "Red Mesa Tank Farm", "kind": "tank farm", "area": "South"},
    {"site_id": "RM-CS", "name": "Red Mesa Compressor Station", "kind": "compressor station", "area": "South"},
    {"site_id": "WF-GP", "name": "Willow Flats Gas Plant", "kind": "gas plant", "area": "East"},
    {"site_id": "DF-WF", "name": "Dry Fork Water Facility", "kind": "water facility", "area": "East"},
    {"site_id": "PR-PY", "name": "Pine Ridge Pipe Yard", "kind": "pipe yard", "area": "West"},
    {"site_id": "EP-FO", "name": "Eagle Point Field Office", "kind": "office", "area": "West"},
]

SEVERITY_POLICY = """Severity levels
- low: nobody hurt; any spill is under 50 litres and stays contained; no fire; minor damage.
- medium: first aid only; a spill of 50 to 500 litres, or any spill that leaves its containment;
  a small fire put out on site; damage that takes a piece of equipment out of service.
- high: an injury that needs a doctor; a spill over 500 litres; a fire that needs the fire
  department; a gas release.
- critical: a life-threatening injury; a spill that reaches a creek or river; an explosion; the
  site is evacuated.
When a report fits two levels, pick the higher one.

Escalate to the HSE manager when any of these is true:
- anyone was hurt, even first aid only
- there was a fire, even one put out at once
- the severity is high or critical"""



def lookup_site(name: str) -> str:
    """Find a site in the registry by name. Returns the closest site."""
    matches = [s for s in SITES if name.casefold() in s["name"].casefold()]
    if len(matches) > 1:
        return f"More than one site matches {name!r}. Try again with the full site name."
    if not matches:
        closest = get_close_matches(name, [s["name"] for s in SITES], n=1, cutoff=0)
        matches = [s for s in SITES if s["name"] == closest[0]]
    return json.dumps(matches[0])


def get_severity_policy() -> str:
    """Get the full severity policy document. Only needed for unusual cases."""
    return SEVERITY_POLICY




class Triage(BaseModel):
    incident_type: Literal["injury", "spill", "fire", "near_miss", "equipment", "vehicle"]
    severity: Literal["low", "medium", "high", "critical"]
    site: str = Field(description='The site\'s name from the registry, or "unknown"')
    injury: bool = Field(description="Was anyone hurt?")
    escalate: bool = Field(description="Should the HSE manager hear about it now?")
    reason: str = Field(description="One sentence on why this severity")


SYSTEM_PROMPT = """You triage HSE (health, safety and environment) incident reports from the field
sites of Juniper Basin Energy.

For each report:
1. Find the site with lookup_site. Every triage must name a site from the registry.
2. Classify the incident and set its severity from the reporter's written description. Photos
   are often from earlier incidents, so go by the text.
3. Decide whether to escalate to the HSE manager."""

agent_v0 = create_agent(
    model=MODEL,
    tools=[lookup_site, get_severity_policy],
    system_prompt=SYSTEM_PROMPT,
    response_format=Triage,
    name="hse-triage",
)

---

## 3. One evaluator per problem

Each evaluator below exists because of a problem you found in the hunt. None of them needs a
"correct answer" for each example, so the same checks could also watch production traffic.

| Problem from the hunt | Evaluator | Kind |
|---|---|---|
| The photo shows a fire or a spill, the severity says "low" | `severity_matches_photo` | LLM judge that sees the photo |
| The output names a site the report never mentions | `site_in_report` | Code |
| The agent skips the severity policy | `expected_steps` | Trajectory match |
| The agent calls `lookup_site` over and over | `expected_steps`, plus the token and cost columns | Trajectory match |
| A 👍 on a wrong answer | none: feedback is not a label | |

First, two helpers that pull the report and the photo out of an example's input.

In [ ]:
def report_of(inputs: dict) -> str:
    content = inputs["messages"][0]["content"]
    return " ".join(block["text"] for block in content if block["type"] == "text")


def photo_of(inputs: dict) -> str | None:
    content = inputs["messages"][0]["content"]
    return next((block["base64"] for block in content if block["type"] == "image"), None)

### Is the site in the report? (code)

When the check is exact, write code. It's free, fast, and never unsure.

In [ ]:
def site_in_report(inputs: dict, outputs: dict) -> bool:
    site = outputs["triage"]["site"]
    return site == "unknown" or site.casefold() in report_of(inputs).casefold()

### Did the agent take the expected steps? (trajectory)

Every report needs one site lookup and one look at the policy, in any order. `agentevals`
compares the tool calls the agent made with that expected path. `unordered` mode matches calls
one to one, so a skipped policy check fails, and so do repeated lookups.

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator
from langchain.messages import AIMessage

EXPECTED_STEPS = [
    AIMessage("", tool_calls=[{"name": "lookup_site", "args": {}, "id": "1"}]),
    AIMessage("", tool_calls=[{"name": "get_severity_policy", "args": {}, "id": "2"}]),
]

match_steps = create_trajectory_match_evaluator(
    trajectory_match_mode="unordered", tool_args_match_mode="ignore"
)


def expected_steps(outputs: dict) -> dict:
    result = match_steps(outputs=outputs["messages"], reference_outputs=EXPECTED_STEPS)
    return {"score": result["score"]}

### Does the severity fit the photo? (LLM judge)

Code can't look at a photo, so this one takes a model. The judge gets the severity levels, the
agent's severity and the photo, and answers yes or no with a reason. Reports without a photo
get a comment and no score.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


class Verdict(BaseModel):
    fits: bool = Field(description="Does the agent's severity fit what the photo shows?")
    reason: str


JUDGE_PROMPT = """You check the work of an agent that triages HSE incident reports.

{policy}

The agent rated this incident: {severity}

Look at the photo the reporter attached. Does it fit that severity? Answer no if the photo
shows a worse incident than the rating, such as a fire, smoke or a large spill rated "low"."""

judge = init_chat_model(MODEL).with_structured_output(Verdict)


def severity_matches_photo(inputs: dict, outputs: dict) -> dict:
    photo = photo_of(inputs)
    if not photo:
        return {"comment": "No photo to check."}
    prompt = JUDGE_PROMPT.format(policy=SEVERITY_POLICY, severity=outputs["triage"]["severity"])
    verdict = judge.invoke([HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image", "base64": photo, "mime_type": "image/jpeg"},
    ])])
    return {"score": verdict.fits, "comment": verdict.reason}

---

## 4. Score v0

An **experiment** runs the agent on every example in the dataset, then runs each evaluator on
the result. `target` turns an agent into the function an experiment calls: it takes an
example's input and returns the `Triage` plus the messages, which `expected_steps` needs.

In [ ]:
def target(agent):
    def run(inputs: dict) -> dict:
        result = agent.invoke(inputs)
        return {
            "triage": result["structured_response"].model_dump(),
            "messages": result["messages"],
        }

    return run


EVALUATORS = [site_in_report, expected_steps, severity_matches_photo]

v0 = client.evaluate(
    target(agent_v0),
    data=DATASET,
    evaluators=EVALUATORS,
    experiment_prefix="v0",
    max_concurrency=4,
)

---

## 5. The fix: v1

Each change answers one problem. Some are in the tools, the rest are in the prompt.

| Problem | Change |
|---|---|
| Photo ignored | The prompt says to look at the photo, and that the photo wins |
| Invented site | `lookup_site` returns no match instead of the closest site; the prompt says to use "unknown" |
| Skipped policy | The prompt says to always call `get_severity_policy`, and its docstring no longer calls it optional |
| Repeated lookups | `lookup_site` lists every match; the prompt says to call it once |

In [ ]:
def lookup_site(name: str) -> str:
    """Find sites in the registry by the name the report uses. Lists every match."""
    query = name.casefold()
    matches = [s for s in SITES if query in s["name"].casefold() or s["name"].casefold() in query]
    return json.dumps(matches) if matches else 'No site matches. Use "unknown".'


def get_severity_policy() -> str:
    """Get the severity levels and the rules for escalating to the HSE manager."""
    return SEVERITY_POLICY


SYSTEM_PROMPT_V1 = """You triage HSE (health, safety and environment) incident reports from the field
sites of Juniper Basin Energy.

For each report:
1. Call lookup_site once, with the place the report mentions. If nothing matches, or more than
   one site matches and the report doesn't say which, the site is "unknown".
2. Always call get_severity_policy, and set the severity and escalation by its rules.
3. If a photo is attached, look at it. When it shows something worse than the text (fire,
   smoke, a large spill), the photo wins."""

agent_v1 = create_agent(
    model=MODEL,
    tools=[lookup_site, get_severity_policy],
    system_prompt=SYSTEM_PROMPT_V1,
    response_format=Triage,
    name="hse-triage",
)

---

## 6. Score v1 and compare

In [ ]:
v1 = client.evaluate(
    target(agent_v1),
    data=DATASET,
    evaluators=EVALUATORS,
    experiment_prefix="v1",
    max_concurrency=4,
)

print("Compare v0 and v1:", f"{v1.comparison_url},{v0.experiment_id}")

Open the link. For each evaluator, the comparison view shows v1's score next to v0's, and
highlights the examples that got better or worse. Check the **tokens** and **cost** columns
too: that's where the repeated lookups show up.

---

## 📌 Key takeaways

- Start from traces, not from a list of metrics. Every evaluator here exists because of a
  problem you found.
- Use code when the check is exact, such as the site or the steps. Use a judge only when it
  takes judgment, such as reading a photo.
- A trajectory evaluator checks how the agent got its answer, not only the answer.
- None of these evaluators needs a labeled answer, so the same checks can watch production.
- The comparison view shows which examples got better or worse, not just the averages.